# Gaia Astrometric Periodogram
Modify the LSP so it can find astrometric signals in Gaia DR4 astrometric residuals, fast.

## authors:
- **David W. Hogg** (Flatiron)

## contributors:

## projects:
- Demonstrate that we have a project.

## bugs & to-dos:
- Show that the four amplitudes can be used to characterize the companion signal.
- Need to simultaneously fit position, parallax, and proper motion (just add cpsi, spsi, and some others to the fit).

In [ ]:
# !pip install --upgrade pip
# !pip install --upgrade astropy # this project requires Astropy 8.0.2

In [ ]:
import numpy as np
import pylab as plt
from astropy.timeseries import LombScargle
from astropy.timeseries.periodograms.lombscargle.implementations.utils import trig_sum

In [ ]:
rng = np.random.default_rng(17) # the most random number

In [ ]:
# make *extremely unrealistic* toy fake data
N = 128
times = np.sort(rng.uniform(size=N)) * 5. * 365.25 # 5 years in days
period = 1.3 * np.pi * 365.25 # 1.3 * pi years in days
omega = 2. * np.pi / period
phi_0 = 5.2331 # radians
scan_angles = rng.uniform(size=N) * 2. * np.pi # one circle in radians
a = 2.155 # arbitrary units
psi_0 = 1.1 # radians
b = 1.3 # arbitrary units; same as a
thetas = np.array([ a * np.cos(psi_0), a * np.sin(psi_0)])[None, :] * np.cos(omega * times[:, None] - phi_0) \
       + np.array([-b * np.sin(psi_0), b * np.cos(psi_0)])[None, :] * np.sin(omega * times[:, None] - phi_0)
ivars = 3. + 1.5 * rng.uniform(size=N)
errs = 1. / np.sqrt(ivars)
noises = rng.normal(size=N) * errs
ys = np.sum(thetas * np.vstack([np.cos(scan_angles - psi_0), np.sin(scan_angles - psi_0)]).T, axis=1)

In [ ]:
# check the (noiseless) internals of the generative model
plt.figure(figsize=(4, 4))
plt.scatter(thetas[:, 0], thetas[:, 1], c=times)
plt.axis("equal")

In [ ]:
# check that the data look "okay"
plt.figure(figsize=(8, 4))
plt.errorbar(times, ys, marker="none", color="k", linestyle="none", yerr=errs)
plt.scatter(times, ys, c=scan_angles)

In [ ]:
def slow_angle_periodogram(t, psi, y, iv, f, rcond=None):
    """
    ## bugs:
    - No `fit_mean` input.
    - Does not fit for position, parallax, or proper motion (these are easy to include).
    - Should be a jax vmap, not a for loop.
    """
    cpsi, spsi = np.cos(psi), np.sin(psi)
    tpt = 2. * np.pi * t
    pars = np.zeros((len(f), 4))
    chisq = np.zeros(len(f))
    for i, ff in enumerate(np.atleast_1d(f)):
        ct, st = np.cos(ff * tpt), np.sin(ff * tpt)
        X = np.vstack([ct * cpsi, st * cpsi, ct * spsi, st * spsi]).T
        pars[i], _, _, _ = np.linalg.lstsq(X.T @ (iv[:,None] * X), X.T @ (iv * y), rcond=rcond)
        resid = y - X @ pars[i]
        chisq[i] = np.sum(iv * resid * resid)
    return pars, chisq

In [ ]:
fs = np.arange(1. / 3652.5, 1. / (1.2 * 365.25), 0.25 / (3652.5)) # 1.2 years to 10.0 years, over-sampled
pars, chisquareds = slow_angle_periodogram(times, scan_angles, ys, ivars, fs)
ibest = np.argmin(chisquareds)
print("best pars on grid:", pars[ibest], "at period", 1. / fs[ibest])

In [ ]:
plt.axvline(period, c="g")
plt.plot(1. / fs, chisquareds, "k-")

In [ ]:
bestpars = pars[ibest]
x_extent = np.sqrt(bestpars[0] ** 2 + bestpars[1] ** 2)
y_extent = np.sqrt(bestpars[2] ** 2 + bestpars[3] ** 2)
b_over_a = np.sin(2. * np.atan2(bestpars[1], bestpars[0]) - np.atan2(bestpars[3], bestpars[2]))
print(x_extent, y_extent, b_over_a, b / a)